# 01 — De notícias a features

**Objetivo:** transformar notícias não estruturadas (`1 linha = 1 notícia`) em sinais numéricos diários (`1 linha = 1 dia`) que os modelos conseguem utilizar.

### Entrada

`data/silver/news/br_energy_news.csv`

### Saída

`data/gold/features/br_energy_news_features_daily.csv`

### Features iniciais

- `news_count`
- `signal`
- `avg_score`
- `avg_relevance`
- `positive_count`
- `neutral_count`
- `negative_count`
- `positive_share`
- `neutral_share`
- `negative_share`
- `max_relevance`
- `score_std`
- `signal_ma_3`
- `signal_ma_5`
- `signal_ma_7`

O `signal` diário segue a lógica:

```text
Σ(score × relevance)
────────────────────
    Σ(relevance)
```


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

NEWS_PATH = PROJECT_ROOT / "data/silver/news/br_energy_news.csv"
SCORES_PATH = PROJECT_ROOT / "data/silver/nlp/br_energy_news_scores.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/gold/features/br_energy_news_features_daily.csv"

print("News input :", NEWS_PATH)
print("Scores input:", SCORES_PATH)
print("Output     :", OUTPUT_PATH)


News input : c:\Users\e702746\projetos_main\Can-AI-listen-to-the-market\data\silver\news\br_energy_news.csv
Scores input: c:\Users\e702746\projetos_main\Can-AI-listen-to-the-market\data\silver\nlp\br_energy_news_scores.csv
Output     : c:\Users\e702746\projetos_main\Can-AI-listen-to-the-market\data\gold\features\br_energy_news_features_daily.csv


## 1. Carregar as notícias


In [2]:
news = pd.read_csv(NEWS_PATH)
scores = pd.read_csv(SCORES_PATH)
scores['_row_order'] = range(len(scores))
scores['_analyzed_at'] = pd.to_datetime(
    scores['analyzed_at'], utc=True, errors='coerce'
)
scores = (
    scores.sort_values(
        ['article_id', '_analyzed_at', '_row_order'],
        na_position='first',
    )
    .drop_duplicates('article_id', keep='last')
    .drop(columns=['_row_order', '_analyzed_at'])
)
news = news.merge(scores, on='article_id', how='inner', validate='one_to_one')

news["published_at"] = pd.to_datetime(
    news["published_at"],
    utc=True,
    errors="coerce",
)

news["score"] = pd.to_numeric(
    news["score"],
    errors="coerce",
)

news["relevance"] = pd.to_numeric(
    news["relevance"],
    errors="coerce",
)

print(f"Rows: {len(news):,}")
print(f"Period: {news['published_at'].min()} -> {news['published_at'].max()}")

news.head()


Rows: 2,375
Period: 2026-01-05 11:46:41+00:00 -> 2026-10-23 17:35:58+00:00


,article_id,source,title,summary,url,published_at,fetched_at,category,sentiment,score,relevance,reason,model,prompt_version,analyzed_at
0,000f0fdd49cd71b79e699cd29ea37f51171d2350486822...,megawhat,"Neoenergia revoga mais uma usina, de 51 MW, en...",A Agência Nacional de Energia Elétrica (Aneel)...,https://megawhat.uol.com.br/geracao/neoenergia...,2026-03-10 14:13:38+00:00,2026-08-25T14:36:30+00:00,energy_br,neutral,0.0,0.12,"A revogação de uma usina solar de 51,2 MW pode...",gpt-5.4-mini,energy-price-next-session-v1,NaN
1,00145cafb0be3e5bc25a77f748a56a9b834608b9906dd6...,megawhat,🎙️EPP fora do LRCap e mais uma comercializador...,Jornalista Camila Maia explica a não habilitaç...,https://megawhat.uol.com.br/podcasts/epp-fora-...,2026-05-29 13:09:04+00:00,2026-08-25T14:11:20+00:00,energy_br,neutral,0.0,0.12,A notícia trata de temas corporativos e regula...,gpt-5.4-mini,energy-price-next-session-v1,NaN
2,008d2ab5afcd7a3ec5b5c94e58d9e54c0dc7768c13fc31...,megawhat,CPFL Soluções avalia preços voláteis e janela ...,A crescente participação de geração renovável ...,https://megawhat.uol.com.br/conteudo-de-marca/...,2026-01-12 20:21:57+00:00,2026-08-25T14:53:40+00:00,energy_br,neutral,0.0,0.18,A notícia é genérica e apenas comenta a maior ...,gpt-5.4-mini,energy-price-next-session-v1,NaN
3,00f0d6a3503f8b40567553284e4c97c7cb74f1cfc25929...,megawhat,🎙️Os caminhos jurídicos da caducidade da Enel ...,"No MinutoMega desta terça-feira, 3 de março, a...",https://megawhat.uol.com.br/podcasts/minutomeg...,2026-03-03 12:18:33+00:00,2026-08-25T14:38:58+00:00,energy_br,neutral,0.0,0.12,A notícia trata בעיקר de um processo jurídico ...,gpt-5.4-mini,energy-price-next-session-v1,NaN
4,00faa06f50e51b78c22dfb268f9309a36d4fb7d56bba5e...,megawhat,Mineradora de terras raras brasileira recebe f...,Confira as notícias mais relevantes veiculadas...,https://megawhat.uol.com.br/megaexpresso/miner...,2026-02-06 11:50:04+00:00,2026-08-25T14:45:46+00:00,energy_br,neutral,0.0,0.02,A notícia trata de financiamento para uma mine...,gpt-5.4-mini,energy-price-next-session-v1,NaN


## 2. Filtrar notícias analisadas

Para gerar features de sentimento, consideramos somente notícias com `score` e `relevance` disponíveis.

In [3]:
analyzed = news.dropna(
    subset=["published_at", "score", "relevance"]
).copy()

analyzed["date"] = analyzed["published_at"].dt.date

print(f"Analyzed news: {len(analyzed):,}")
print(f"Unique days  : {analyzed['date'].nunique():,}")

analyzed[
    [
        "date",
        "source",
        "title",
        "sentiment",
        "score",
        "relevance",
    ]
].head()


Analyzed news: 2,375
Unique days  : 164


,date,source,title,sentiment,score,relevance
0,2026-03-10,megawhat,"Neoenergia revoga mais uma usina, de 51 MW, en...",neutral,0.0,0.12
1,2026-05-29,megawhat,🎙️EPP fora do LRCap e mais uma comercializador...,neutral,0.0,0.12
2,2026-01-12,megawhat,CPFL Soluções avalia preços voláteis e janela ...,neutral,0.0,0.18
3,2026-03-03,megawhat,🎙️Os caminhos jurídicos da caducidade da Enel ...,neutral,0.0,0.12
4,2026-02-06,megawhat,Mineradora de terras raras brasileira recebe f...,neutral,0.0,0.02


## 3. Contribuição ponderada por notícia

Cada notícia contribui para o sinal diário por:

`weighted_score = score × relevance`

A relevância funciona como peso na agregação.

In [4]:
analyzed["weighted_score"] = (
    analyzed["score"]
    * analyzed["relevance"]
)

analyzed[
    [
        "date",
        "title",
        "score",
        "relevance",
        "weighted_score",
    ]
].head()


,date,title,score,relevance,weighted_score
0,2026-03-10,"Neoenergia revoga mais uma usina, de 51 MW, en...",0.0,0.12,0.0
1,2026-05-29,🎙️EPP fora do LRCap e mais uma comercializador...,0.0,0.12,0.0
2,2026-01-12,CPFL Soluções avalia preços voláteis e janela ...,0.0,0.18,0.0
3,2026-03-03,🎙️Os caminhos jurídicos da caducidade da Enel ...,0.0,0.12,0.0
4,2026-02-06,Mineradora de terras raras brasileira recebe f...,0.0,0.02,0.0


## 4. Features diárias

Agregamos as notícias por `published_at`.

A granularidade final passa a ser diária


In [5]:
sentiment_daily = (
    analyzed
    .assign(
        sentiment=analyzed["sentiment"]
        .astype(str)
        .str.lower()
    )
    .groupby(
        ["date", "sentiment"]
    )
    .size()
    .unstack(fill_value=0)
)

for column in [
    "positive",
    "neutral",
    "negative",
]:
    if column not in sentiment_daily.columns:
        sentiment_daily[column] = 0

sentiment_daily = sentiment_daily[
    [
        "positive",
        "neutral",
        "negative",
    ]
].rename(
    columns={
        "positive": "positive_count",
        "neutral": "neutral_count",
        "negative": "negative_count",
    }
)

sentiment_daily.head()


sentiment,positive_count,neutral_count,negative_count
date,,,
2026-01-05,0,12,2
2026-01-06,0,12,1
2026-01-07,0,14,0
2026-01-08,1,11,0
2026-01-09,0,12,0


In [6]:
daily = (
    analyzed
    .groupby("date")
    .agg(
        news_count=("score", "count"),
        weighted_sum=("weighted_score", "sum"),
        total_relevance=("relevance", "sum"),
        avg_score=("score", "mean"),
        avg_relevance=("relevance", "mean"),
        max_relevance=("relevance", "max"),
        score_std=("score", "std"),
    )
    .join(sentiment_daily)
    .reset_index()
)

daily["signal"] = np.where(
    daily["total_relevance"] > 0,
    daily["weighted_sum"] / daily["total_relevance"],
    0.0,
)

daily["positive_share"] = (
    daily["positive_count"] / daily["news_count"]
)

daily["neutral_share"] = (
    daily["neutral_count"] / daily["news_count"]
)

daily["negative_share"] = (
    daily["negative_count"] / daily["news_count"]
)

daily["score_std"] = daily["score_std"].fillna(0.0)

daily.head()


,date,news_count,weighted_sum,total_relevance,avg_score,avg_relevance,max_relevance,score_std,positive_count,neutral_count,negative_count,signal,positive_share,neutral_share,negative_share
0,2026-01-05,14,-0.3008,2.99,-0.037857,0.213571,0.63,0.109064,0,12,2,-0.100602,0.000000,0.857143,0.142857
1,2026-01-06,13,-0.1240,1.82,-0.011538,0.140000,0.55,0.074032,0,12,1,-0.068132,0.000000,0.923077,0.076923
2,2026-01-07,14,0.0534,1.78,0.022857,0.127143,0.26,0.024939,0,14,0,0.030000,0.000000,1.000000,0.000000
3,2026-01-08,12,0.1886,1.76,0.031667,0.146667,0.62,0.080547,1,11,0,0.107159,0.083333,0.916667,0.000000
4,2026-01-09,12,0.0410,1.46,0.012500,0.121667,0.32,0.031079,0,12,0,0.028082,0.000000,1.000000,0.000000


## 5. Médias móveis

As médias móveis ajudam a reduzir ruído diário e capturar persistência do sinal.

Começamos com janelas simples de 3, 5 e 7 dias de observação.


In [7]:
daily = daily.sort_values("date").reset_index(drop=True)

for window in [3, 5, 7]:
    daily[f"signal_ma_{window}"] = (
        daily["signal"]
        .rolling(
            window=window,
            min_periods=1,
        )
        .mean()
    )

daily[
    [
        "date",
        "signal",
        "signal_ma_3",
        "signal_ma_5",
        "signal_ma_7",
    ]
].tail(10)


,date,signal,signal_ma_3,signal_ma_5,signal_ma_7
154,2026-08-14,-0.120915,-0.045202,-0.080257,-0.144543
155,2026-08-17,0.055164,-0.052959,-0.027277,-0.113731
156,2026-08-18,0.033333,-0.010806,-0.009422,-0.044684
157,2026-08-19,0.216449,0.101649,0.018181,0.016199
158,2026-08-20,0.236977,0.162253,0.084202,0.058045
159,2026-08-21,-0.177526,0.091967,0.072879,0.021480
160,2026-08-24,-0.065420,-0.001990,0.048763,0.025437
161,2026-10-21,0.009701,-0.077748,0.044036,0.044097
162,2026-10-22,0.044935,-0.003594,0.009733,0.042636
163,2026-10-23,0.019149,0.024595,-0.033832,0.040609


## 6. Dataset final

In [8]:
feature_columns = [
    "date",
    "news_count",
    "signal",
    "avg_score",
    "avg_relevance",
    "max_relevance",
    "score_std",
    "positive_count",
    "neutral_count",
    "negative_count",
    "positive_share",
    "neutral_share",
    "negative_share",
    "signal_ma_3",
    "signal_ma_5",
    "signal_ma_7",
]

daily_features = daily[feature_columns].copy()

numeric_columns = daily_features.select_dtypes(
    include="number"
).columns

daily_features[numeric_columns] = (
    daily_features[numeric_columns]
    .round(4)
)

daily_features.tail(10)


,date,news_count,signal,avg_score,avg_relevance,max_relevance,score_std,positive_count,neutral_count,negative_count,positive_share,neutral_share,negative_share,signal_ma_3,signal_ma_5,signal_ma_7
154,2026-08-14,19,-0.1209,-0.0147,0.1495,0.68,0.1419,1,17,1,0.0526,0.8947,0.0526,-0.0452,-0.0803,-0.1445
155,2026-08-17,18,0.0552,0.0250,0.1356,0.45,0.0503,1,17,0,0.0556,0.9444,0.0000,-0.0530,-0.0273,-0.1137
156,2026-08-18,11,0.0333,0.0182,0.1036,0.27,0.0337,0,11,0,0.0000,1.0000,0.0000,-0.0108,-0.0094,-0.0447
157,2026-08-19,13,0.2164,0.0485,0.2123,0.92,0.2064,1,11,1,0.0769,0.8462,0.0769,0.1016,0.0182,0.0162
158,2026-08-20,13,0.2370,0.2108,0.6946,0.89,0.4077,7,4,2,0.5385,0.3077,0.1538,0.1623,0.0842,0.0580
159,2026-08-21,13,-0.1775,-0.1123,0.5877,0.92,0.4403,2,6,5,0.1538,0.4615,0.3846,0.0920,0.0729,0.0215
160,2026-08-24,22,-0.0654,-0.0364,0.7473,0.95,0.4553,5,8,9,0.2273,0.3636,0.4091,-0.0020,0.0488,0.0254
161,2026-10-21,9,0.0097,0.0111,0.1489,0.22,0.0220,0,9,0,0.0000,1.0000,0.0000,-0.0777,0.0440,0.0441
162,2026-10-22,6,0.0449,0.0367,0.1283,0.18,0.0497,0,6,0,0.0000,1.0000,0.0000,-0.0036,0.0097,0.0426
163,2026-10-23,12,0.0191,0.0167,0.1175,0.18,0.0246,0,12,0,0.0000,1.0000,0.0000,0.0246,-0.0338,0.0406


## 7. Validações rápidas

- uma linha por dia;
- `signal` entre -1 e +1;
- shares entre 0 e 1;
- contagem de sentimentos igual a `news_count`.


In [9]:
assert daily_features["date"].is_unique

assert daily_features["signal"].between(
    -1,
    1,
).all()

for column in [
    "positive_share",
    "neutral_share",
    "negative_share",
]:
    assert daily_features[column].between(
        0,
        1,
    ).all()

sentiment_total = (
    daily_features["positive_count"]
    + daily_features["neutral_count"]
    + daily_features["negative_count"]
)

assert (
    sentiment_total
    == daily_features["news_count"]
).all()

print("✓ Validations passed")


✓ Validations passed


## 8. Persistir features diárias

`02_market_news_eda.ipynb`


In [10]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

daily_features.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(
    f"✓ Saved {len(daily_features):,} daily rows "
    f"to {OUTPUT_PATH}"
)


✓ Saved 164 daily rows to c:\Users\e702746\projetos_main\Can-AI-listen-to-the-market\data\gold\features\br_energy_news_features_daily.csv


## 9. Resumo

O dataset final contém uma linha por dia com:

- intensidade e direção agregada das notícias;
- relevância média;
- volume de notícias;
- distribuição de sentimento;
- dispersão dos scores;
- médias móveis do sinal.


In [ ]:
daily_features.head()

,date,news_count,signal,avg_score,avg_relevance,max_relevance,score_std,positive_count,neutral_count,negative_count,positive_share,neutral_share,negative_share,signal_ma_3,signal_ma_5,signal_ma_7
0,2026-01-05,14,-0.1006,-0.0379,0.2136,0.63,0.1091,0,12,2,0.0000,0.8571,0.1429,-0.1006,-0.1006,-0.1006
1,2026-01-06,13,-0.0681,-0.0115,0.1400,0.55,0.0740,0,12,1,0.0000,0.9231,0.0769,-0.0844,-0.0844,-0.0844
2,2026-01-07,14,0.0300,0.0229,0.1271,0.26,0.0249,0,14,0,0.0000,1.0000,0.0000,-0.0462,-0.0462,-0.0462
3,2026-01-08,12,0.1072,0.0317,0.1467,0.62,0.0805,1,11,0,0.0833,0.9167,0.0000,0.0230,-0.0079,-0.0079
4,2026-01-09,12,0.0281,0.0125,0.1217,0.32,0.0311,0,12,0,0.0000,1.0000,0.0000,0.0551,-0.0007,-0.0007


: 